In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
path = '/content/drive/My Drive/Colab_Notebooks/NLP'
os.chdir(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Importing Libraries

**pip install only on colab**

In [ ]:
#pip install langchain-text-splitters

In [ ]:
#pip install bitsandbytes

In [ ]:
#pip install faiss-cpu

In [ ]:
from datasets import load_dataset, get_dataset_split_names, get_dataset_config_names, load_dataset_builder, Dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from tqdm import tqdm
import numpy as np
from pathlib import Path
import re
import torch.nn.functional as F
from torch import Tensor
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss

# Inspecting Dataset

**Reference**:
_MTS, Department of War UAP Release 1 — structured corpus, 2026. CC-BY-4.0._
This dataset is a structured, machine-readable companion to the source material at [war.gov/UFO/](https://www.war.gov/UFO/).

In [ ]:
ufo_dataset = "MTSLIVE/war-gov-uap-release-1"
get_dataset_config_names(ufo_dataset)   # dataset files

We will use the `pages` file.

In [ ]:
pages = load_dataset(ufo_dataset, "pages", split="train")
pages

Dataset({
    features: ['document_id', 'page_no', 'text', 'has_figures'],
    num_rows: 4239
})

In [ ]:
# example of what we are going to use
print(pages[0]['text'])

HEADQUARTERS
AIR MATERIEL COMMAND
WRIGHT FIELD, DAYTON, OHIO

DEC 1 9 1947

SUBJECT: Flying Discs

TO: Chief of Staff
United States Air Force
Washington 25, D. C.
ATTENTION: Director, Research & Development
Major General L. C. Craigie

1. Confirming the recent conversation of the undersigned with Major General L. C. Craigie, 9 December 1947, attached as listed below are copies of the reports from this Headquarters concerning Flying Discs.

2. Comments of Headquarters, Air Force on these letters have never been received by this Command. Continued and recent reports from qualified observers concerning this phenomenon still makes this matter one of concern to Headquarters, Air Materiel Command. Intelligence Department of this Command is continuing the collection and analysis of all available reports.

FOR THE COMMANDING GENERAL:

H. M. McCOY
Colonel, USAF
Chief of Intelligence

2 Attach:
cc ltr to CG, AAF, dtd 23 Sept 47 subj "AMC Opinion Concerning "Flying Discs""
cc ltr to CG, AAF, dtd 

# Check some documents by id

In [ ]:
# all documents, each of these is composed of 1 or more pages
IDS = set(pages["document_id"])

Let's see if some pages have less than 20 characters.

In [ ]:
null_id = 0     # count for document_id (even if it's just one page)
null_pgs = 0    # count for pages (can share the same id)
doc_ids = []    # to filter later(?)

for page in pages:
    if len(page["text"]) <= 20:
        null_pgs+=1
        if page["document_id"] not in doc_ids:
            null_id+=1
            doc_ids.append(page["document_id"])
        #print(f"doc_id: {page["document_id"]}\n text: {page["text"]}", "\n")

print(f"total null ids (at least one page): {null_id}")
print(f"total null pages: {null_pgs}")
print(f"problematic ids: {doc_ids}")

So there are 61 `document_id` with _at least_ one page with less than 20 characters. If we talk in terms of pages, there are 225 pages almost empty.

In [ ]:
count = pages.to_pandas().groupby("document_id").count()
single = list(count[count["page_no"]==1].index)  # Documents of 1 page only
# Print the single paged documents
# for doc in pages.filter(lambda x: x["document_id"] in single): print(f"DOCUMENT: {doc["document_id"].upper()}\n\n{doc["text"]}\n\n\n\n")

In [ ]:
# Discard single paged documents containing no information
pattern = re.compile("fbi-photo|nasa-uap-vm|fbi-september-2023-sighting-composite-sketch")  # compile pattern to match discarded documents
useIDS = set(ID for ID in IDS if not pattern.match(ID))  # discard matches

In [ ]:
filterPages = pages.filter(lambda x: x["document_id"] in useIDS and x["text"] != '')

# Embedding

The embedder is choosen from [here](https://huggingface.co/spaces/mteb/leaderboard)


**[Qwen3](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) Model Architecture**: <br>
is designed using dual-encoder and cross-encoder architectures the Embedding model processes a single text segment as input, extracting the semantic representation by utilizing the hidden state vector corresponding to the final [EOS] token. ([ref](https://qwen.ai/blog?id=qwen3-embedding))

<div style="display:flex; gap:20px;">
    <img src="https://miro.medium.com/v2/resize:fit:750/format:webp/1*jzZ_e5Bmvx84zPEa-LhqDQ.png"
         alt="Qwen3 architecture"
         width="330"/
         height="180">
    <img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*AWPQxx4xpiQGCp5XNNfyYA.png"
         alt="BERT vs Qwen3"
         width="350"/
         height="180">
</div>

[article](https://arxiv.org/pdf/2506.05176)
For text embeddings, we utilize LLMs with causal attention, appending an
[EOS] token at the end of the input sequence. The final embedding is derived from the hidden state of the last layer corresponding to this [EOS] token.
To ensure embeddings follow instructions during downstream tasks, we concatenate the instruction and the query into a single input context, while leaving the document unchanged before processing with LLMs. The input format for queries is as follows:
`{Instruction}{Query}<|endoftext|>`

In poche parole: vogliamo tensori che hanno lo stesso numero di token → facciamo padding = aggiungiamo dei [PAD] tokens per riempire lo spazio che manca. Attenzione: il pad lo facciamo a sinistra perché il modello alla fine della fiera si basa sull'ultimo token (EOS) che dovrebbe aver compresso in sè il significato di tutt l'input (l'input è dato da: task-accoppiata a-quesry + documenti), quindi a destra devo lasciare il token [EOS] che prelevo. Questi [PAD] vengono filtrati grazie alla `attention_mask` (1=token vero, 0=è un PAD lascia stare).

In [ ]:
# defining the embedding model and the relative tokenizer
# for the tokenizer we need the left padding
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

# harrier
tokenizer = AutoTokenizer.from_pretrained("microsoft/harrier-oss-v1-0.6b", padding_side='left', cache_dir='tokenizers_cache')
model = AutoModel.from_pretrained("microsoft/harrier-oss-v1-0.6b", attn_implementation={"text_config": "flash_attention2"}, quantization_config=quantization_config, cache_dir='models_cache')

# qwen
# We recommend enabling flash_attention_2 for better acceleration and memory saving.
#tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-4B', padding_side='left', cache_dir='tokenizers_cache')
#model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-4B', quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")  #  attn_implementation="flash_attention_2" not installed

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Per ora stiamo:
- dividendo il testo di ogni pagina in chunk (la divisione avviene con il limite di 512 token e l'overlap di 77 ma è "interna", l'output sono dei chunk di testo, non di token)
- mettendo insieme (come previsto) task-query-chunk_di_testo
- dandoli come input al tokenizer il cui output sono degli effettivi token (paddati)
- dandoli come input al modello (che non worka perché la RAM muore)

Il punto è: a noi ci serve LangChain se vogliamo la divisione con overlap ma ha senso dare chunk come input? o forse sarebbe meglio task-query-**pagina**_di_testo? tanto il limite è di 8192 tokens.

In [ ]:
# function to create the chunked text (input of Qwen3 embedder)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=512, chunk_overlap=77)

pages_chunked = []    # list of dictionaries

for page in filterPages:
    page_text = page['text']
    chunk_list = text_splitter.split_text(page_text)
    for i, chunk in enumerate(chunk_list):
        pages_chunked.append({
            'document_id': page['document_id'],
            'page_no': page['page_no'],
            'chunk_id': i,
            'chunk_text': chunk
    })

chunkedPages = Dataset.from_list(pages_chunked)

In [ ]:
chunkedPages

Dataset({
    features: ['document_id', 'page_no', 'chunk_id', 'chunk_text'],
    num_rows: 5321
})

In [ ]:
# function to extract the last token (EOS) that is a compressed representation of the whole input (instruction+query)
def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

# we "merge" in one string instruction and query
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery: {query}'

In [ ]:
def get_embeddings(text_list, max_length=512):

  # Tokenize the input texts
  batch_dict = tokenizer(
    text_list,
    padding='longest',
    truncation=True,
    max_length=max_length,
    return_tensors="pt",
  )
  batch_dict.to(model.device)

  with torch.no_grad(): outputs = model(**batch_dict)
  embeddings = last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask']).cpu()

  torch.cuda.empty_cache()

  return embeddings

In [ ]:
docsEmbed = torch.load("harrier06DocsEmbed.pt", weights_only=False)

In [ ]:
embedPages = chunkedPages.add_column("embeddings", docsEmbed.to(torch.float32).cpu().numpy().tolist())

In [ ]:
embedPages

Dataset({
    features: ['document_id', 'page_no', 'chunk_id', 'chunk_text', 'embeddings'],
    num_rows: 5321
})

### Queries

Here we embed only the queries. We need to add the documents part (that here we just load)
```python
# input_texts = chunks[:160]  # batchsize 80 for qwen3-0.6 40 for qwen3-4b  ## done in .py script
```

In [ ]:
# Each query must come with a one-sentence instruction that describes the task
task = 'Given a document search query, retrieve relevant passages that answer the query'

raw_queries = [
    "What are Jesus's powers?",
    'What was spotted in the sky for the first time?',
    'Have UFOs ever been close to humans (astronauts)?',
    "What patterns emerge across the reported UAP sightings regarding location, altitude, behavior, speed, and time period?",
    "What are the most extravagant sightings?",
    "What is the Saucer's secret?",
    "What are they hiding from us?"
]

queries = [get_detailed_instruct(task, query) for query in raw_queries]

In [ ]:
queriesEmbed = get_embeddings(queries)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


---
## Faiss part - using Harrier06

with `add_faiss_index` we could directly access the dataset so keeping the metadata (and by-passing the numpydocs-numpyqueries thing) but there is an issue in using this method that i don't know how to solve atm

In [ ]:
numpydocs = docsEmbed.to(torch.float32).cpu().numpy()
numpyqueries = queriesEmbed.to(torch.float32).cpu().numpy()
index = faiss.index_factory(docsEmbed.shape[1], "Flat", faiss.METRIC_INNER_PRODUCT)   # cosine similarity
index.ntotal
faiss.normalize_L2(numpydocs)
index.add(numpydocs)
faiss.normalize_L2(numpyqueries)

k=5
k_best_distance, k_best_index = index.search(numpyqueries, k)
#print('Distances:{}'.format(k_best_distance))

In [ ]:
for i, query in enumerate(raw_queries):
    best_idx = int(k_best_index[i, 0])
    print(f"Query: {query}")
    print(f"Best chunk: {embedPages[best_idx]['chunk_text']}")
    print(f"Score: {k_best_distance[i, 0]:.4f}")
    print("---")

Query: What are Jesus's powers?
Best chunk: The life of Jesus, now too, becomes clearer when these things are re-membered. His powers of levitation, His ability to pass through doors, walk on water, and heal the sick, are the essential attributes of men from outer space. They will also be ours someday when we will be "free like birds" (Ezk. 13:20). At Jesus' birth the celestial army came quite close to earth. A space-man appeared to the shepherds and the "glory" of the Lord, with the usual signs of His presence, shone around them. There was a multitude of the heavenly army with this space-man. And after their cosmic announcement, the music of their space-ships was heard as they again disappeared into space.

Jesus' ascension is described as "a cloud (or space-ship) received Him out of their sight" (Acts 1:9). His coming again is to be in the same manner. "Then will appear the sign of the Son of man in heaven (space) coming on the clouds (space-ships) of heaven (space) with power and gr

# Generation

In [ ]:
embedQue = embeddings
embedDocs = torch.load("/mnt/eph/embedding/qwen4DocsEmbed.pt", weights_only=False)
embeddings = torch.cat([embedQue, embedDocs])
embeddingsNorm = F.normalize(embeddings, p=2, dim=1)
# normalize embeddings
scores = (embeddingsNorm[:len(queries)] @ embeddingsNorm[len(queries):].T)*100
topVals, topIdx = scores.topk(5, dim=1)  # topIdx are the pages_chunked idxes
# print(scores.tolist())

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

# harrier
# tokenizer = AutoTokenizer.from_pretrained("microsoft/harrier-oss-v1-0.6b", padding_side='left', cache_dir='tokenizers_cache')
# model = AutoModel.from_pretrained("microsoft/harrier-oss-v1-0.6b", attn_implementation={"text_config": "flash_attention2"}, quantization_config=quantization_config, cache_dir='models_cache')

# qwen
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B', padding_side='left', cache_dir='tokenizers_cache')
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B', quantization_config=quantization_config, attn_implementation={"text_config": "flash_attention2"}, cache_dir="models_cache")

In [ ]:
# FROM DOCS https://huggingface.co/Qwen/Qwen3-4B more or less
system_prompt = "We are impartial detectives investigating the latest FBI UAP encounters. Use the provided documents to answer the Query as best as you can."
for i, query in enumerate(queries):
    ## through pages_chunked[idx] you can access any metainformation of the document. It is not necessary possibly even useless to feed it to the generator
    retrDocs = [pages_chunked[idx]["chunk_text"] for idx in 4[i]]
    prompt = "Provided documents:\n"
    for j, doc in enumerate(retrDocs): prompt += f"Document {str(j + 1)}:\n{doc}.\n\n"
    prompt += f"\nQuery: {query}"
    print(prompt)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=8192
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print("thinking content:", thinking_content)
    print("content:", content)
    print()